# Apache Spark + Delta Lake 🔺

**Trabalho de Pesquisa — Engenharia de Dados (SATC)**
Autores: **Murilo Mandelli** e **Augusto Benedetti**

Este notebook demonstra, com PySpark, a criação de tabelas no formato **Delta Lake** e os comandos
**INSERT**, **UPDATE** e **DELETE** (além de MERGE e *time travel*) sobre um cenário de seguradora.

**Roteiro**
1. Cenário e modelo ER
2. Ambiente (SparkSession com Delta)
3. DDL — criação das tabelas
4. Carga inicial a partir dos CSV
5. INSERT
6. UPDATE
7. DELETE
8. MERGE (upsert)
9. Histórico, *time travel* e RESTORE
10. Por dentro do Delta: o `_delta_log`
11. Consulta analítica final


## 1. Cenário

Uma **seguradora de veículos** precisa manter o cadastro de clientes, carros, apólices e sinistros
em um *lakehouse* local. Usamos o mesmo conjunto de dados do **laboratório de Modelagem
Dimensional** da disciplina (`script_relacional.sql`, banco SQL Server), convertido para CSV pelo
script `scripts/converter_sql_para_csv.py`.

![Modelo ER](../assets/modelo_er.svg)

| Tabela | Linhas | Descrição | Chave |
|---|---:|---|---|
| `cliente` | 20.010 | Pessoas que contratam seguros | `cd_cliente` |
| `carro` | 10.001 | Veículos segurados | `placa` |
| `apolice` | 10.000 | Contrato de seguro: liga **1 cliente** a **1 carro**, com vigência, cobertura e franquia | `cd_apolice` |
| `sinistro` | 10.000 | Ocorrências (acidentes/roubos) envolvendo um carro | `cd_sinistro` |

**Relacionamentos:** `cliente 1 ── N apolice N ── 1 carro 1 ── N sinistro`.

> O Spark **não impõe** chaves primárias/estrangeiras: elas são *lógicas* e servem para os JOINs.


## 2. Ambiente

A função `criar_sessao_delta()` (em `src/spark_delta_iceberg/spark_session.py`) cria a
`SparkSession` com:

- `spark.sql.extensions = io.delta.sql.DeltaSparkSessionExtension` → habilita `UPDATE`, `DELETE`, `MERGE`, `DESCRIBE HISTORY`...
- `spark.sql.catalog.spark_catalog = org.apache.spark.sql.delta.catalog.DeltaCatalog` → o catálogo padrão passa a entender `USING DELTA`
- `configure_spark_with_delta_pip(...)` → baixa o JAR do Delta compatível com o pacote `delta-spark`

As tabelas são gravadas em `data/lakehouse/delta/`. Apagamos essa pasta no início para que a
demonstração sempre comece do zero (reprodutível).

In [1]:
import shutil
from importlib.metadata import version

from delta.tables import DeltaTable

from spark_delta_iceberg import TABELAS, criar_sessao_delta, ler_csv
from spark_delta_iceberg.config import PASTA_DELTA

shutil.rmtree(PASTA_DELTA, ignore_errors=True)

spark = criar_sessao_delta()
print("Apache Spark:", spark.version)
print("delta-spark :", version("delta-spark"))
print("Warehouse   :", PASTA_DELTA)

26/10/05 21:14:00 WARN Utils: Your hostname, DESKTOP-OM56K2B resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/05 21:14:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/root/spark-delta-iceberg/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-08b7ba97-c7ab-4b35-a960-1ec619f82798;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central


	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 216ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.3 from central in [default]
	io.delta#delta-storage;3.3.3 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-08b7ba97-c7ab-4b35-a960-1ec619f82798
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/9ms)


26/10/05 21:14:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Apache Spark: 3.5.9
delta-spark : 3.3.3
Warehouse   : /root/spark-delta-iceberg/data/lakehouse/delta


## 3. DDL — criação das tabelas

Criamos o *database* `seguradora` e as 4 tabelas com `USING DELTA`. Os tipos seguem o modelo
relacional do laboratório (`DECIMAL(10,2)` para valores, `TIMESTAMP` para datas com hora).

In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS seguradora")
spark.sql("USE seguradora")

spark.sql("""
CREATE TABLE IF NOT EXISTS seguradora.cliente (
    cd_cliente     INT     NOT NULL COMMENT 'PK - código do cliente',
    nome           STRING           COMMENT 'Nome completo',
    cpf            STRING  NOT NULL COMMENT 'CPF (texto para manter zeros à esquerda)',
    sexo           STRING           COMMENT 'M/F',
    dt_nascimento  DATE             COMMENT 'Data de nascimento'
) USING DELTA
COMMENT 'Clientes da seguradora'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS seguradora.carro (
    placa      STRING NOT NULL COMMENT 'PK - placa do veículo',
    cd_modelo  INT             COMMENT 'Código do modelo',
    chassi     STRING NOT NULL COMMENT 'Número do chassi',
    ano        INT             COMMENT 'Ano de fabricação',
    cor        STRING          COMMENT 'Cor do veículo'
) USING DELTA
COMMENT 'Veículos segurados'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS seguradora.apolice (
    cd_apolice          INT           NOT NULL COMMENT 'PK - número da apólice',
    cd_cliente          INT           NOT NULL COMMENT 'FK -> cliente',
    dt_inicio_vigencia  TIMESTAMP              COMMENT 'Início da vigência',
    dt_fim_vigencia     TIMESTAMP              COMMENT 'Fim da vigência',
    vl_cobertura        DECIMAL(10,2)          COMMENT 'Valor de cobertura (R$)',
    vl_franquia         DECIMAL(10,2)          COMMENT 'Valor da franquia (R$)',
    placa               STRING                 COMMENT 'FK -> carro'
) USING DELTA
COMMENT 'Apólices: ligam 1 cliente a 1 carro'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS seguradora.sinistro (
    cd_sinistro     INT       NOT NULL COMMENT 'PK - código do sinistro',
    placa           STRING    NOT NULL COMMENT 'FK -> carro',
    dt_sinistro     TIMESTAMP          COMMENT 'Data/hora da ocorrência',
    local_sinistro  INT                COMMENT 'Código IBGE do município',
    condutor        STRING             COMMENT 'Nome do condutor'
) USING DELTA
COMMENT 'Sinistros (ocorrências) por veículo'
""")

spark.sql("SHOW TABLES IN seguradora").show()

+----------+---------+-----------+
| namespace|tableName|isTemporary|
+----------+---------+-----------+
|seguradora|  apolice|      false|
|seguradora|    carro|      false|
|seguradora|  cliente|      false|
|seguradora| sinistro|      false|
+----------+---------+-----------+



In [3]:
# Detalhes de uma tabela: formato, localização e propriedades
spark.sql("DESCRIBE DETAIL seguradora.apolice").select(
    "format", "name", "location", "numFiles", "minReaderVersion", "minWriterVersion"
).show(truncate=False)

+------+--------------------------------+-------------------------------------------------------------------------+--------+----------------+----------------+
|format|name                            |location                                                                 |numFiles|minReaderVersion|minWriterVersion|
+------+--------------------------------+-------------------------------------------------------------------------+--------+----------------+----------------+
|delta |spark_catalog.seguradora.apolice|file:/root/spark-delta-iceberg/data/lakehouse/delta/seguradora.db/apolice|0       |1               |2               |
+------+--------------------------------+-------------------------------------------------------------------------+--------+----------------+----------------+



## 4. Carga inicial

Lemos cada CSV com **schema explícito** (`src/spark_delta_iceberg/dados.py`) e gravamos nas tabelas
com a API de DataFrame `writeTo(...).append()` — que é, na prática, um **INSERT em lote**.

In [4]:
for tabela in TABELAS:
    ler_csv(spark, tabela).writeTo(f"seguradora.{tabela}").append()
    print(f"{tabela:<9} {spark.table(f'seguradora.{tabela}').count():>6} linhas")

cliente    20010 linhas


carro      10001 linhas


apolice    10000 linhas


sinistro   10000 linhas


In [5]:
spark.table("seguradora.apolice").show(5)

+----------+----------+-------------------+-------------------+------------+-----------+-------+
|cd_apolice|cd_cliente| dt_inicio_vigencia|    dt_fim_vigencia|vl_cobertura|vl_franquia|  placa|
+----------+----------+-------------------+-------------------+------------+-----------+-------+
|         1|      4795|2025-02-01 15:25:41|2025-09-09 15:25:41|   554382.00|   77298.00|HEG0631|
|         2|      6043|2023-09-23 06:57:10|2024-06-13 06:57:10|   453879.00|   16425.00|ILM4667|
|         3|     11754|2023-12-03 05:46:51|2024-03-25 05:46:51|   991737.00|   70757.00|QNR4589|
|         4|      1521|2024-02-08 11:45:33|2024-05-14 11:45:33|   432802.00|   87636.00|LWE5122|
|         5|     17956|2023-07-21 06:41:54|2023-10-08 06:41:54|   100988.00|   76181.00|ZDF8957|
+----------+----------+-------------------+-------------------+------------+-----------+-------+
only showing top 5 rows



## 5. INSERT

`INSERT INTO ... VALUES` acrescenta linhas. No Delta, cada comando gera uma **nova versão** da
tabela (um novo arquivo JSON no `_delta_log`) — a operação é **atômica (ACID)**: ou grava tudo,
ou nada.

Cenário: dois novos clientes contratam seguro; um deles já registra um sinistro em Criciúma
(código IBGE `4204608`).

In [6]:
spark.sql("""
INSERT INTO seguradora.cliente VALUES
    (20011, 'ANA PAULA SOUZA',     '12345678901', 'F', DATE'1995-04-12'),
    (20012, 'CARLOS EDUARDO LIMA', '10987654321', 'M', DATE'1988-11-30')
""")

spark.sql("INSERT INTO seguradora.carro VALUES ('SAT2026', 10, '9BWZZZ377VT004251', 2024, 'PRATA')")

spark.sql("""
INSERT INTO seguradora.apolice VALUES
    (10001, 20011, TIMESTAMP'2026-10-01 00:00:00', TIMESTAMP'2027-10-01 00:00:00',
     85000.00, 4500.00, 'SAT2026')
""")

spark.sql("""
INSERT INTO seguradora.sinistro VALUES
    (10001, 'SAT2026', TIMESTAMP'2026-10-03 14:30:00', 4204608, 'ANA PAULA')
""")

spark.sql("SELECT * FROM seguradora.cliente WHERE cd_cliente >= 20011").show(truncate=False)
spark.sql("SELECT * FROM seguradora.apolice WHERE cd_apolice = 10001").show()

+----------+-------------------+-----------+----+-------------+
|cd_cliente|nome               |cpf        |sexo|dt_nascimento|
+----------+-------------------+-----------+----+-------------+
|20012     |CARLOS EDUARDO LIMA|10987654321|M   |1988-11-30   |
|20011     |ANA PAULA SOUZA    |12345678901|F   |1995-04-12   |
+----------+-------------------+-----------+----+-------------+



+----------+----------+-------------------+-------------------+------------+-----------+-------+
|cd_apolice|cd_cliente| dt_inicio_vigencia|    dt_fim_vigencia|vl_cobertura|vl_franquia|  placa|
+----------+----------+-------------------+-------------------+------------+-----------+-------+
|     10001|     20011|2026-10-01 00:00:00|2027-10-01 00:00:00|    85000.00|    4500.00|SAT2026|
+----------+----------+-------------------+-------------------+------------+-----------+-------+



## 6. UPDATE

`UPDATE` altera linhas existentes. O Delta **não edita** arquivos Parquet: ele reescreve os arquivos
afetados e registra no log quais saíram (`remove`) e quais entraram (`add`).

**Exemplo 1 (SQL)** — o mesmo caso do laboratório: o carro `CXK4218` foi pintado de **PRETO**.

In [7]:
print("ANTES")
spark.sql("SELECT placa, ano, cor FROM seguradora.carro WHERE placa = 'CXK4218'").show()

spark.sql("UPDATE seguradora.carro SET cor = 'PRETO' WHERE placa = 'CXK4218'")

print("DEPOIS")
spark.sql("SELECT placa, ano, cor FROM seguradora.carro WHERE placa = 'CXK4218'").show()

ANTES


+-------+----+------+
|  placa| ano|   cor|
+-------+----+------+
|CXK4218|2008|CHUMBO|
+-------+----+------+



DEPOIS


+-------+----+-----+
|  placa| ano|  cor|
+-------+----+-----+
|CXK4218|2008|PRETO|
+-------+----+-----+



**Exemplo 2 (SQL, várias linhas)** — reajuste de **10%** na cobertura das apólices iniciadas em 2025.

In [8]:
consulta = """
SELECT COUNT(*) AS qtd_apolices, SUM(vl_cobertura) AS total_cobertura
FROM seguradora.apolice WHERE YEAR(dt_inicio_vigencia) = 2025
"""
print("ANTES");  spark.sql(consulta).show()

spark.sql("""
UPDATE seguradora.apolice
SET vl_cobertura = CAST(vl_cobertura * 1.10 AS DECIMAL(10,2))
WHERE YEAR(dt_inicio_vigencia) = 2025
""").show()  # o Delta devolve o número de linhas afetadas

print("DEPOIS"); spark.sql(consulta).show()

ANTES


+------------+---------------+
|qtd_apolices|total_cobertura|
+------------+---------------+
|         982|   550415493.00|
+------------+---------------+



+-----------------+
|num_affected_rows|
+-----------------+
|              982|
+-----------------+

DEPOIS


+------------+---------------+
|qtd_apolices|total_cobertura|
+------------+---------------+
|         982|   605457042.30|
+------------+---------------+



**Exemplo 3 (API Python `DeltaTable`)** — o Delta também oferece uma API orientada a objetos.
Corrigimos o nome de um cliente:

In [9]:
tb_cliente = DeltaTable.forName(spark, "seguradora.cliente")

tb_cliente.update(
    condition="cd_cliente = 20012",
    set={"nome": "'CARLOS EDUARDO LIMA FILHO'"},
)
spark.sql("SELECT * FROM seguradora.cliente WHERE cd_cliente = 20012").show(truncate=False)

+----------+-------------------------+-----------+----+-------------+
|cd_cliente|nome                     |cpf        |sexo|dt_nascimento|
+----------+-------------------------+-----------+----+-------------+
|20012     |CARLOS EDUARDO LIMA FILHO|10987654321|M   |1988-11-30   |
+----------+-------------------------+-----------+----+-------------+



## 7. DELETE

`DELETE` remove linhas que atendem a uma condição. Os dados antigos **continuam no disco** até um
`VACUUM`, por isso ainda é possível consultá-los via *time travel*.

**Exemplo 1 (SQL)** — um sinistro foi lançado por engano.

In [10]:
spark.sql("DELETE FROM seguradora.sinistro WHERE cd_sinistro = 10001").show()
spark.sql("SELECT COUNT(*) AS qtd FROM seguradora.sinistro WHERE cd_sinistro = 10001").show()

+-----------------+
|num_affected_rows|
+-----------------+
|                1|
+-----------------+



+---+
|qtd|
+---+
|  0|
+---+



**Exemplo 2 (SQL, várias linhas)** — expurgo das apólices encerradas antes de julho/2023.
Guardamos a versão atual para restaurar depois.

In [11]:
tb_apolice = DeltaTable.forName(spark, "seguradora.apolice")
versao_antes_delete = tb_apolice.history(1).first()["version"]

print("Total ANTES :", spark.table("seguradora.apolice").count())
spark.sql("DELETE FROM seguradora.apolice WHERE dt_fim_vigencia < TIMESTAMP'2023-07-01 00:00:00'").show()
print("Total DEPOIS:", spark.table("seguradora.apolice").count())

Total ANTES : 10001


+-----------------+
|num_affected_rows|
+-----------------+
|              201|
+-----------------+



Total DEPOIS: 9800


**Exemplo 3 (API Python)** — o cliente 20012 desistiu antes de contratar.

In [12]:
tb_cliente.delete("cd_cliente = 20012")
spark.sql("SELECT * FROM seguradora.cliente WHERE cd_cliente >= 20011").show(truncate=False)

+----------+---------------+-----------+----+-------------+
|cd_cliente|nome           |cpf        |sexo|dt_nascimento|
+----------+---------------+-----------+----+-------------+
|20011     |ANA PAULA SOUZA|12345678901|F   |1995-04-12   |
+----------+---------------+-----------+----+-------------+



## 8. MERGE (upsert)

`MERGE` combina INSERT e UPDATE num único comando atômico: atualiza quem já existe e insere quem é
novo. É o mesmo padrão usado no laboratório para a carga **SCD Tipo 1** da `dim_cliente`.

In [13]:
atualizacoes = spark.createDataFrame(
    [
        (20011, "ANA PAULA SOUZA MARTINS", "12345678901", "F", "1995-04-12"),  # já existe -> UPDATE
        (20013, "BRUNO HENRIQUE COSTA",    "11122233344", "M", "2001-02-20"),  # novo      -> INSERT
    ],
    "cd_cliente INT, nome STRING, cpf STRING, sexo STRING, dt_nascimento STRING",
).selectExpr("cd_cliente", "nome", "cpf", "sexo", "CAST(dt_nascimento AS DATE) AS dt_nascimento")
atualizacoes.createOrReplaceTempView("cliente_atualizacoes")

spark.sql("""
MERGE INTO seguradora.cliente AS destino
USING cliente_atualizacoes AS origem
ON destino.cd_cliente = origem.cd_cliente
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""").show()

spark.sql("SELECT * FROM seguradora.cliente WHERE cd_cliente >= 20011").show(truncate=False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|                2|               1|               0|                1|
+-----------------+----------------+----------------+-----------------+



+----------+-----------------------+-----------+----+-------------+
|cd_cliente|nome                   |cpf        |sexo|dt_nascimento|
+----------+-----------------------+-----------+----+-------------+
|20011     |ANA PAULA SOUZA MARTINS|12345678901|F   |1995-04-12   |
|20013     |BRUNO HENRIQUE COSTA   |11122233344|M   |2001-02-20   |
+----------+-----------------------+-----------+----+-------------+



## 9. Histórico, *time travel* e RESTORE

Cada operação virou uma versão. `DESCRIBE HISTORY` mostra quem fez o quê:

In [14]:
spark.sql("DESCRIBE HISTORY seguradora.carro").selectExpr(
    "version",
    "timestamp",
    "operation",
    "operationMetrics['numOutputRows']  AS linhas_gravadas",
    "operationMetrics['numUpdatedRows'] AS linhas_atualizadas",
).show(truncate=False)

+-------+-----------------------+------------+---------------+------------------+
|version|timestamp              |operation   |linhas_gravadas|linhas_atualizadas|
+-------+-----------------------+------------+---------------+------------------+
|3      |2026-10-05 21:14:37.362|UPDATE      |NULL           |1                 |
|2      |2026-10-05 21:14:31.042|WRITE       |1              |NULL              |
|1      |2026-10-05 21:14:23.708|WRITE       |10001          |NULL              |
|0      |2026-10-05 21:14:13.414|CREATE TABLE|NULL           |NULL              |
+-------+-----------------------+------------+---------------+------------------+



Com `VERSION AS OF` consultamos a tabela **como ela era** antes do UPDATE (cor original do carro):

In [15]:
spark.sql("""
SELECT 'versão 1 (carga)' AS versao, placa, cor FROM seguradora.carro VERSION AS OF 1 WHERE placa = 'CXK4218'
UNION ALL
SELECT 'versão atual'     AS versao, placa, cor FROM seguradora.carro WHERE placa = 'CXK4218'
""").show()

+----------------+-------+------+
|          versao|  placa|   cor|
+----------------+-------+------+
|versão 1 (carga)|CXK4218|CHUMBO|
|    versão atual|CXK4218| PRETO|
+----------------+-------+------+



`RESTORE` volta a tabela para uma versão anterior — aqui desfazemos o expurgo de apólices:

In [16]:
print("Antes do RESTORE:", spark.table("seguradora.apolice").count())
spark.sql(f"RESTORE TABLE seguradora.apolice TO VERSION AS OF {versao_antes_delete}").show()
print("Depois do RESTORE:", spark.table("seguradora.apolice").count())

Antes do RESTORE: 9800


+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|table_size_after_restore|num_of_files_after_restore|num_removed_files|num_restored_files|removed_files_size|restored_files_size|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|                  497340|                         2|                1|                 1|            484794|             494942|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+



Depois do RESTORE: 10001


## 10. Por dentro do Delta: o `_delta_log`

Uma tabela Delta é uma pasta com arquivos **Parquet** (dados) + a pasta **`_delta_log`** com um
JSON por versão (o *transaction log*). Vejamos os arquivos da tabela `carro` e o commit do UPDATE:

In [17]:
import json

pasta_carro = PASTA_DELTA / "seguradora.db" / "carro"
for arquivo in sorted((pasta_carro / "_delta_log").glob("*.json")):
    print(arquivo.name)

print("\nCommit 3 (UPDATE) — ações registradas:")
with open(pasta_carro / "_delta_log" / "00000000000000000003.json") as f:
    for linha in f:
        acao = json.loads(linha)
        tipo = next(iter(acao))
        detalhe = acao[tipo].get("path") or acao[tipo].get("operation") or ""
        print(f"  {tipo:<12} {detalhe}")

00000000000000000000.json
00000000000000000001.json
00000000000000000002.json
00000000000000000003.json

Commit 3 (UPDATE) — ações registradas:
  commitInfo   UPDATE
  add          part-00000-78d4e721-88c3-4b9c-8adf-4b5664ee0844-c000.snappy.parquet
  remove       part-00000-c8622bf6-438b-43e9-99e4-fbec2eb62762-c000.snappy.parquet


## 11. Consulta analítica final

As tabelas continuam consultáveis normalmente com SQL e JOINs. Top 5 clientes com mais sinistros:

In [18]:
spark.sql("""
SELECT c.cd_cliente, c.nome, COUNT(s.cd_sinistro) AS qtd_sinistros
FROM seguradora.cliente  c
JOIN seguradora.apolice  a ON a.cd_cliente = c.cd_cliente
JOIN seguradora.sinistro s ON s.placa      = a.placa
GROUP BY c.cd_cliente, c.nome
ORDER BY qtd_sinistros DESC, c.cd_cliente
LIMIT 5
""").show(truncate=False)

+----------+----------------+-------------+
|cd_cliente|nome            |qtd_sinistros|
+----------+----------------+-------------+
|1406      |YURI VIEIRA     |5            |
|3263      |LUCCA MARTINS   |5            |
|8756      |RENAN RAMOS     |5            |
|18658     |CAROLINE DA ROSA|5            |
|185       |MAYSA RODRIGUES |4            |
+----------+----------------+-------------+



In [19]:
spark.stop()

## Conclusão

- Criamos 4 tabelas **Delta** com DDL SQL e carregamos ~50 mil linhas a partir de CSV.
- **INSERT**, **UPDATE**, **DELETE** e **MERGE** funcionaram via SQL e via API `DeltaTable`.
- Cada operação gerou uma versão no `_delta_log`, permitindo **auditoria** (`DESCRIBE HISTORY`),
  **time travel** (`VERSION AS OF`) e **RESTORE**.

Documentação completa: veja o site MkDocs do projeto.